# Módulo 3: Big Data y la Analítica de Datos para la Toma de Decisiones
## Unidad 1: Preparación y Arquitectura de Datos
### **Notebook Formativo 2: Ingesta, ETL, Pandas y Datos Faltantes**

**Carga, Inspección y Limpieza de Datasets Industriales**
**Diplomado:** Gestión Industrial y Analítica de Datos — FCyT UMSS

---

## Propósito
Este notebook formativo guía al alumno paso a paso en la implementación de un pipeline básico de Extracción, Transformación y Carga (ETL) utilizando la biblioteca Pandas en Google Colab.

Al finalizar esta sesión, usted podrá:
1. Cargar un archivo de datos industriales aplicando parámetros explícitos (separadores, codificación, decimales).
2. Inspeccionar la estructura física, los tipos inferidos y la completitud del dataset.
3. Identificar discrepancias entre el tipo de datos inferido por el software y el contrato esperado por el negocio.
4. Diagnosticar, cuantificar y tratar los valores faltantes aplicando criterios justificados del proceso de planta, evitando la mala práctica de rellenar con ceros automáticamente.

## 0. Inicialización y simulación de datos sucios de planta

Para que este laboratorio sea completamente autónomo y funcional en su entorno de Colab, primero vamos a simular el archivo crudo `produccion_industrial.csv`. Este archivo representa el típico caos IT/OT: fechas mezcladas, lecturas nulas, decimales con coma, números cargados como texto y códigos centinela de fallos mecánicos (como el `-999` que representa un cortocircuito en el sensor).

In [20]:
# Simulación de la ingesta de planta
#este ejerccio se hizo con la version de python 3.13.15 y pandas 2.2.3

data_dirty = """timestamp;equipo;temp_c;energia_kwh;unidades;estado
2026-08-11 07:00;E-07;742.3;118.4;52;OK
11/08/2026 07:05;e07;-999;120,1;51;OK
2026-08-11T07:10Z;E-07;"745,2";—;50;alerta
2026-08-11 07:10;E-07;745.2;121.0;50;ALERTA
2026/08/11 07:15;E-7;744.1;119.8;cincuenta;OK"""

with open("produccion_industrial.csv", "w", encoding="utf-8") as f:
    f.write(data_dirty.strip())

print("Archivo 'produccion_industrial.csv' creado localmente en el runtime de Colab.")
import sys
print(sys.version)
import pandas as pd
print(pd.__version__)

Archivo 'produccion_industrial.csv' creado localmente en el runtime de Colab.
3.13.15 (tags/v3.13.15:4061bc4, Aug  5 2026, 13:05:39) [MSC v.1944 64 bit (AMD64)]
2.2.3


## 1. Carga de Datos (Ingesta con Intención)

Un error muy común es cargar el archivo usando simplemente `pd.read_csv(ruta)`. En entornos reales de planta, esto produce tablas deformadas porque el formato regional, la codificación o los caracteres de ausencia varían.

Cargaremos los datos aplicando parámetros explícitos para respetar el contrato de ingesta:

In [21]:
import pandas as pd

# Carga explícita indicando separadores, decimales y caracteres que representan nulos
df_raw = pd.read_csv(
    "produccion_industrial.csv",
    sep=";",
    decimal=",",
    na_values=["", "NA", "—"],
    encoding="utf-8"
)

print(f"Dimensiones iniciales: {df_raw.shape[0]} filas por {df_raw.shape[1]} columnas.\n")
df_raw

Dimensiones iniciales: 5 filas por 6 columnas.



,timestamp,equipo,temp_c,energia_kwh,unidades,estado
0,2026-08-11 07:00,E-07,742.3,118.4,52,OK
1,11/08/2026 07:05,e07,-999,"120,1",51,OK
2,2026-08-11T07:10Z,E-07,"745,2",NaN,50,alerta
3,2026-08-11 07:10,E-07,745.2,121.0,50,ALERTA
4,2026/08/11 07:15,E-7,744.1,119.8,cincuenta,OK


## 2. Inspección Estructural y Radiografía Técnica

Antes de aplicar cualquier filtro o cálculo matemático, debemos inspeccionar los tipos de datos que Pandas ha inferido automáticamente y el nivel de completitud de cada variable.

In [22]:
# Radiografía técnica de dtypes y memoria
print("=== df_raw.info() ===")
df_raw.info()

print("\n=== Tipos de datos inferidos ===")
print(df_raw.dtypes)

print("\n=== df_raw.describe() ===")
df_raw.describe(include="all")


=== df_raw.info() ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   timestamp    5 non-null      object
 1   equipo       5 non-null      object
 2   temp_c       5 non-null      object
 3   energia_kwh  4 non-null      object
 4   unidades     5 non-null      object
 5   estado       5 non-null      object
dtypes: object(6)
memory usage: 372.0+ bytes

=== Tipos de datos inferidos ===
timestamp      object
equipo         object
temp_c         object
energia_kwh    object
unidades       object
estado         object
dtype: object

=== df_raw.describe() ===


,timestamp,equipo,temp_c,energia_kwh,unidades,estado
count,5,5,5,4,5,5
unique,5,3,5,4,4,3
top,2026-08-11 07:00,E-07,742.3,118.4,50,OK
freq,1,3,1,1,2,3


### Pausa de Análisis

Observe detenidamente la columna `temp_c`. En la tabla impresa en el paso 1, vemos que contiene números como `742.3`. Sin embargo, en `df_raw.info()`, su tipo de dato es `object` (cadena de texto).

**¿A qué se debe esto? Identifique qué elemento en la columna está forzando a Pandas a interpretar la temperatura como texto en lugar de número, y qué consecuencias tiene esto si queremos calcular el promedio de temperatura.**

> **Respuesta del participante:**
> Diagnóstico del tipo de dato de temp_c

La columna mezcla temperaturas con punto decimal, como 742.3, y con coma decimal, como 745,2. Como la lectura utiliza decimal=",", no interpreta de forma uniforme todos los valores y pandas 2.2.3 conserva la columna como object.

El problema es la mezcla de formatos, no las comillas del CSV. Antes de calcular el promedio debemos unificar el separador decimal y convertir la columna a numérica. También debemos excluir el centinela -999, porque representa un fallo del sensor y distorsionaría el promedio.

## 3. Transformación - Limpieza y Conversión Segura

Para operar el modelo, crearemos una copia de trabajo de modo que la evidencia original quede intacta. Limpiaremos y convertiremos de forma explícita las variables clave:

- **`timestamp`:** De texto a fecha y hora con unificación de zona horaria.
- **`temp_c` y `energia_kwh`:** A numérico decimal (forzando errores a valores faltantes con `coerce`).
- **`equipo`:** Normalizado a un identificador canónico — mayúsculas, sin espacios, y con guion y ceros a la izquierda unificados. Un `str.upper()` a secas no alcanza: en los datos crudos el mismo equipo físico aparece escrito como `E-07`, `e07` y `E-7`; sin corregirlo, el pipeline los trataría como tres máquinas distintas.

In [23]:
import re

# Crear copia de trabajo
df = df_raw.copy()

# 1. Conversión de fechas (formato mixto de planta)
df["timestamp_utc"] = pd.to_datetime(
    df["timestamp"],
    format="mixed",
    dayfirst=True,
    errors="coerce",
    utc=True
)

# 2. Conversión segura a numérico (temp_c y energia_kwh)
df["temp_c"] = pd.to_numeric(df["temp_c"], errors="coerce")
df["energia_kwh"] = pd.to_numeric(df["energia_kwh"], errors="coerce")

# 3. Normalización del identificador de equipo a un formato canónico LETRA-NN
equipos_antes_normalizar = df["equipo"].astype("string").str.strip().str.upper().unique().tolist()

def normalizar_id_equipo(valor):
    """Normaliza un identificador de equipo al formato canónico LETRA-NN.

    Corrige el guion faltante ('E07' -> 'E-07') y el cero a la izquierda
    faltante ('E-7' -> 'E-07'). Si el texto no sigue el patrón letra(s) +
    número, se devuelve tal cual, sin inventar una corrección, para que
    quede visible y se revise manualmente.
    """
    texto = str(valor).strip().upper()
    coincidencia = re.match(r"^([A-Z]+)-?(\d+)$", texto)
    if not coincidencia:
        return texto
    letra, numero = coincidencia.groups()
    return f"{letra}-{numero.zfill(2)}"

df["equipo"] = df["equipo"].apply(normalizar_id_equipo).astype("string")

print(f"Identificadores de equipo antes de normalizar:  {equipos_antes_normalizar}")
print(f"Identificadores de equipo después de normalizar: {df['equipo'].unique().tolist()}")

print("\n=== Tipos de datos transformados ===")
print(df.dtypes)
df

Identificadores de equipo antes de normalizar:  ['E-07', 'E07', 'E-7']
Identificadores de equipo después de normalizar: ['E-07']

=== Tipos de datos transformados ===
timestamp                     object
equipo                string[python]
temp_c                       float64
energia_kwh                  float64
unidades                      object
estado                        object
timestamp_utc    datetime64[ns, UTC]
dtype: object


,timestamp,equipo,temp_c,energia_kwh,unidades,estado,timestamp_utc
0,2026-08-11 07:00,E-07,742.3,118.4,52,OK,2026-08-11 07:00:00+00:00
1,11/08/2026 07:05,E-07,-999.0,NaN,51,OK,2026-08-11 07:05:00+00:00
2,2026-08-11T07:10Z,E-07,NaN,NaN,50,alerta,2026-08-11 07:10:00+00:00
3,2026-08-11 07:10,E-07,745.2,121.0,50,ALERTA,2026-08-11 07:10:00+00:00
4,2026/08/11 07:15,E-07,744.1,119.8,cincuenta,OK,2026-08-11 07:15:00+00:00


## 4. Detección y Cuarentena de Duplicados

Antes de tratar valores faltantes, hay una pregunta más básica que el pipeline todavía no respondió: ¿cada fila representa una lectura física distinta, o hay más de una fila describiendo el mismo instante del mismo equipo?

Revisando los datos crudos, dos registros comparten equipo `E-07` y el mismo instante `07:10` — uno de ellos sin el dato de `energia_kwh`. Tratarlos como dos lecturas independientes duplicaría artificialmente ese instante en cualquier promedio o conteo posterior. La regla de negocio es simple: **una llave de lectura (equipo + instante) no debería repetirse**; si se repite, hay un evento a resolver, no un dato nuevo.

In [24]:
# La llave de negocio de una lectura es el equipo y el instante en que se tomó.
# Si esa combinación se repite, no son dos mediciones: es el mismo evento registrado más de una vez.
llave_lectura = ["timestamp_utc", "equipo"]

df["es_duplicado"] = df.duplicated(subset=llave_lectura, keep=False)

print(f"Filas que pertenecen a un grupo de lectura duplicada: {df['es_duplicado'].sum()} de {len(df)}")
df[df["es_duplicado"]]

Filas que pertenecen a un grupo de lectura duplicada: 2 de 5


,timestamp,equipo,temp_c,energia_kwh,unidades,estado,timestamp_utc,es_duplicado
2,2026-08-11T07:10Z,E-07,NaN,NaN,50,alerta,2026-08-11 07:10:00+00:00,True
3,2026-08-11 07:10,E-07,745.2,121.0,50,ALERTA,2026-08-11 07:10:00+00:00,True


### Pausa de Análisis — Duplicados

Note que no usamos `df.drop_duplicates()` directamente. Esa función conservaría la primera fila que encuentre y descartaría el resto sin preguntar cuál tenía mejor información.

**¿Por qué, en un contexto industrial, decidir qué fila conservar según su nivel de completitud es más defendible que quedarse con "la primera que aparece"? ¿Qué información se perdería si hubiéramos hecho lo contrario?**

> **Respuesta del participante:**
> Criterio de conservación de duplicados

Una vez confirmado que los registros representan el mismo equipo y el mismo instante, conservar la fila con más información válida es más defendible que elegir por orden de aparición: ese orden no garantiza calidad.

En este caso, conservar la primera fila de las 07:10 dejaría la energía faltante y descartaría el valor disponible de 121.0 kWh de la otra fila. La temperatura 745,2 de la primera fila sí es recuperable al normalizar el separador decimal.

La fila retirada debe mantenerse en cuarentena con el motivo de la decisión para permitir su revisión. Si dos registros contienen valores válidos contradictorios, la completitud por sí sola no basta para decidir.

In [25]:
# Puntaje de completitud: cuántos campos de negocio están presentes en la fila.
# No usamos "la primera que aparece"; usamos la fila con más evidencia real.
columnas_negocio = ["temp_c", "energia_kwh", "unidades", "estado"]
df["completitud_score"] = df[columnas_negocio].notna().sum(axis=1)

# Dentro de cada grupo duplicado, nos quedamos con la fila más completa.
# El resto no se descarta: se separa a cuarentena para revisión, con su motivo documentado.
indice_a_conservar = (
    df[df["es_duplicado"]]
    .sort_values("completitud_score", ascending=False)
    .drop_duplicates(subset=llave_lectura, keep="first")
    .index
)

es_duplicado_a_retirar = df["es_duplicado"] & ~df.index.isin(indice_a_conservar)

df_cuarentena = df[es_duplicado_a_retirar].copy()
df_cuarentena["motivo_cuarentena"] = "duplicado menos completo de la misma lectura"

df = df[~es_duplicado_a_retirar].drop(columns=["es_duplicado", "completitud_score"]).copy()

print(f"Filas crudas originales: {len(df_raw)}")
print(f"Filas curadas que continúan en el pipeline: {len(df)}")
print(f"Filas en cuarentena por duplicado menos completo: {len(df_cuarentena)}")
df_cuarentena

Filas crudas originales: 5
Filas curadas que continúan en el pipeline: 4
Filas en cuarentena por duplicado menos completo: 1


,timestamp,equipo,temp_c,energia_kwh,unidades,estado,timestamp_utc,es_duplicado,completitud_score,motivo_cuarentena
2,2026-08-11T07:10Z,E-07,NaN,NaN,50,alerta,2026-08-11 07:10:00+00:00,True,2,duplicado menos completo de la misma lectura


Con esto, el dataset queda en el estado que se retoma al abrir la Sesión 3: **cinco filas crudas, cuatro filas curadas disponibles para el análisis, y una fila en cuarentena** por ser el duplicado menos completo de la misma lectura. `df_cuarentena` no se descarta — queda disponible como evidencia, con su motivo documentado, para quien necesite auditar la decisión.

## 5. Identificación y Tratamiento de Valores Faltantes

Una vez convertidos los datos incompatibles a `NaN`, procedemos a cuantificar la tasa de ausencia en nuestro dataset. No podemos ocultar los nulos usando `fillna(0)` mecánicamente, ya que un cero en temperatura representaría congelación física en planta, alterando drásticamente el cálculo posterior.

In [33]:
# Tablero de control de completitud de datos
resumen_calidad = pd.DataFrame({
    "tipo_dato": df.dtypes.astype(str),
    "valores_faltantes": df.isna().sum(),
    "porcentaje_faltante": (df.isna().mean() * 100).round(2)
}).sort_values("porcentaje_faltante", ascending=False)

resumen_calidad
#df

,tipo_dato,valores_faltantes,porcentaje_faltante
energia_kwh,float64,1,25.0
timestamp,object,0,0.0
equipo,string,0,0.0
temp_c,float64,0,0.0
unidades,object,0,0.0
estado,object,0,0.0
timestamp_utc,"datetime64[ns, UTC]",0,0.0
temp_imputada_flag,int64,0,0.0


### Tratamiento Justificado

Aislando los fallos catastróficos del sensor (el valor centinela `-999` de temperatura), aplicaremos una imputación coherente:
1. **Código -999:** Lo convertiremos a `NaN` porque es un error eléctrico, no una lectura de proceso real.
2. **Imputación de temp_c:** Usaremos la mediana histórica calculada por máquina para no sesgar las mediciones.

In [27]:
# 1. Reemplazar código de error centinela -999 por NaN antes de imputar
import numpy as np
df["temp_c"] = df["temp_c"].replace(-999.0, np.nan)

# Crear un indicador de calidad (bandera) que registre la ocurrencia de la imputación
df["temp_imputada_flag"] = df["temp_c"].isna().astype(int)

# 2. Calcular la mediana de temperatura por equipo para imputar
mediana_por_maquina = df.groupby("equipo")["temp_c"].transform("median")
df["temp_c"] = df["temp_c"].fillna(mediana_por_maquina)

df

,timestamp,equipo,temp_c,energia_kwh,unidades,estado,timestamp_utc,temp_imputada_flag
0,2026-08-11 07:00,E-07,742.3,118.4,52,OK,2026-08-11 07:00:00+00:00,0
1,11/08/2026 07:05,E-07,744.1,NaN,51,OK,2026-08-11 07:05:00+00:00,1
3,2026-08-11 07:10,E-07,745.2,121.0,50,ALERTA,2026-08-11 07:10:00+00:00,0
4,2026/08/11 07:15,E-07,744.1,119.8,cincuenta,OK,2026-08-11 07:15:00+00:00,0


### Pausa de Análisis 2

La variable `unidades` posee el valor `'cincuenta'` en formato texto, lo que provocará un error de tipo al integrarse al modelo relacional.

**Escriba una regla lógica utilizando Pandas o una función personalizada que permita normalizar y convertir a entero esta variable sin perder el registro de la fila correspondiente.**

> **Propuesta técnica del participante:**
> Propongo reemplazar el texto conocido cincuenta por 50 y convertir la columna al tipo entero nullable Int64. Los valores no reconocidos quedarán como faltantes para su revisión, sin eliminar la fila. Conservaré el valor original para mantener la trazabilidad.

In [35]:
df["unidades_original"] = df["unidades"].copy()

unidades_normalizadas = (
    df["unidades"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({"cincuenta": "50"})
)

df["unidades"] = pd.to_numeric(
    unidades_normalizadas,
    errors="coerce"
).astype("Int64")
df

,timestamp,equipo,temp_c,energia_kwh,unidades,estado,timestamp_utc,temp_imputada_flag,unidades_original
0,2026-08-11 07:00,E-07,742.3,118.4,52,OK,2026-08-11 07:00:00+00:00,0,52
1,11/08/2026 07:05,E-07,744.1,NaN,51,OK,2026-08-11 07:05:00+00:00,1,51
3,2026-08-11 07:10,E-07,745.2,121.0,50,ALERTA,2026-08-11 07:10:00+00:00,0,50
4,2026/08/11 07:15,E-07,744.1,119.8,50,OK,2026-08-11 07:15:00+00:00,0,50


## 6. Validación y Verificación Final

Un pipeline de datos robusto termina validando que los resultados cumplan con criterios físicos plausibles antes de persistirse en almacenamiento. Aplicaremos comprobaciones automáticas con `assert`.

In [36]:
# Comprobaciones automáticas de integridad física
try:
    # Verificación de que no existen fechas vacías
    assert df["timestamp_utc"].notna().all(), "Error: Existen fechas no convertidas o inválidas."

    # Verificación de que las temperaturas se encuentren en rangos plausibles de operación (0 a 1800°C)
    assert df["temp_c"].dropna().between(0, 1800).all(), "Error: Lecturas de temperatura fuera del límite físico plausibles."

    print("Todas las comprobaciones de calidad han sido superadas con éxito.")
except AssertionError as e:
    print(f"Fallo de validación técnica: {e}")

Todas las comprobaciones de calidad han sido superadas con éxito.


---
## Trabajo Autónomo

Para consolidar esta sesión, cada grupo deberá elaborar la **Ficha del Dataset del Proyecto Integrador** y subirla a la plataforma de Classroom.

Asegúrese de incluir en su informe:
1. La delimitación de la pregunta de negocio y la variable objetivo elegida.
2. La fuente del dato, el responsable de su gestión y la autorización de uso.
3. El diccionario de datos detallando tipos, unidades físicas y límites lógicos de cada sensor.
4. La identificación de riesgos de calidad latentes (ej. centinelas de error de hardware) y su tratamiento justificado.
5. El enlace al Notebook formativo 2 reproducible y debidamente comentado por los participantes.